# Enterprise Text-to-SQL — Benchmark Evaluation & Generation

**Author:** Garvit Audichya ([GitHub](https://github.com/garvit-010/Voice-Text-to-SQL-FineTuning))

This notebook loads the fine-tuned QLoRA adapter (`garvit-010/qwen3-8b-text2sql-qlora-v3`) or base Qwen3-8B and generates predictions for all 453 held-out benchmark questions under greedy decoding.


In [ ]:
!pip install -q transformers>=5.0 peft>=0.20 bitsandbytes>=0.50 accelerate>=1.0 datasets sqlglot huggingface-hub
print('Dependencies installed successfully.')

In [ ]:
import os, json, time, re, hashlib
from pathlib import Path
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), 'GPU required! Enable GPU T4 in Kaggle Settings.'
print(f'Using GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# Locate eval pack
kaggle_input = Path('/kaggle/input')
eval_pack_file = None

for p in kaggle_input.rglob('eval_pack.jsonl'):
    eval_pack_file = p
    break

assert eval_pack_file, 'eval_pack.jsonl not found in /kaggle/input!'
questions = [json.loads(l) for l in eval_pack_file.read_text().splitlines() if l.strip()]
print(f'Loaded {len(questions)} test questions from {eval_pack_file}')

# Load prompt and schema
prompt_file = eval_pack_file.parent / 'prompt_module.py'
schema_file = eval_pack_file.parent / 'schema_context.txt'

import importlib.util
spec = importlib.util.spec_from_file_location('prompt_module', str(prompt_file))
prompt_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prompt_mod)

SCHEMA = schema_file.read_text(encoding='utf-8')
SCHEMA_FP = hashlib.sha256(SCHEMA.encode()).hexdigest()[:16]
PROMPT_FP = prompt_mod.prompt_fingerprint()
print(f'Prompt Fingerprint: {PROMPT_FP} | Schema Fingerprint: {SCHEMA_FP}')


In [ ]:
BASE_MODEL = 'Qwen/Qwen3-8B'
REVISION = 'b968826d9c46'
ADAPTER_ID = 'garvit-010/qwen3-8b-text2sql-qlora-v3'

print('Loading base model in 4-bit NF4...')
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=REVISION,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    ),
    device_map={'': 0},
    low_cpu_mem_usage=True,
)

print(f'Attaching adapter from {ADAPTER_ID}...')
model = PeftModel.from_pretrained(base, ADAPTER_ID).eval()
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Model ready for evaluation!')


In [ ]:
def extract_sql(raw: str) -> str:
    text = re.sub(r'<think>.*?</think>', '', raw, flags=re.S | re.I)
    text = re.sub(r'<think>.*', '', text, flags=re.S | re.I).strip()
    fenced = re.search(r'```(?:sql|postgresql)?\s*(.*?)```', text, re.S | re.I)
    if fenced:
        text = fenced.group(1).strip()
    match = re.search(r'\b(WITH|SELECT)\b', text, re.I)
    if match:
        text = text[match.start():]
    if ';' in text:
        text = text.split(';', 1)[0]
    return ' '.join(text.split()).strip()

output_file = Path('/kaggle/working/predictions_v3.jsonl')
predictions = []
started_all = time.perf_counter()

for i, q in enumerate(questions):
    messages = prompt_mod.build_messages(q['question'], SCHEMA)
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    enc = tokenizer(text, return_tensors='pt', add_special_tokens=False).to(model.device)
    
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    elapsed = (time.perf_counter() - t0) * 1000
    
    raw = tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    sql = extract_sql(raw)
    
    pred = {
        'id': q['id'],
        'question': q['question'],
        'raw_output': raw,
        'predicted_sql': sql,
        'generation_ms': round(elapsed, 1),
        'prompt_fingerprint': PROMPT_FP,
        'schema_fingerprint': SCHEMA_FP,
    }
    predictions.append(pred)
    with open(output_file, 'a', encoding='utf-8') as f:
        f.write(json.dumps(pred) + '\n')
    if (i+1) % 50 == 0 or (i+1) == len(questions):
        print(f'[{i+1}/{len(questions)}] generated in {(time.perf_counter()-started_all)/60:.1f} min')

print(f'Finished generating {len(predictions)} predictions -> {output_file}')
